# Land subsidence with the CSUB package: delay vs. no-delay interbeds

Pumping groundwater lowers the water pressure in an aquifer, which increases the load carried by the sediment skeleton and squeezes the fine-grained, compressible layers - the **interbeds** - so the land surface sinks. This is **land subsidence**, and MODFLOW 6 simulates it with the **Skeletal Storage, Compaction, and Subsidence (CSUB)** package.

This notebook builds a regional aquifer-system compaction model - adapted from MODFLOW 6 example *ex-gwf-csub-p04* - and runs it two ways to compare the two kinds of interbed the CSUB package supports:

- **No-delay interbeds** compact *instantly* to equilibrium whenever the effective stress changes. They are inexpensive and appropriate for thin interbeds that drain quickly.
- **Delay interbeds** drain *slowly*: pore pressure has to diffuse out through the low-permeability clay, so the compaction **lags** behind the change in aquifer head. CSUB resolves the vertical flow inside each interbed with a small one-dimensional grid.

By the end you will build a CSUB model, run it with each interbed type, plot how subsidence develops over time, and **assess the maximum subsidence at the end of the simulation** to quantify the difference.

**A few terms.** As water pressure drops, the **effective stress** (the load on the sediment skeleton) rises. Sediment compacts **elastically** (recoverable) until the effective stress exceeds the largest value it has felt before - the **preconsolidation stress** - after which it compacts **inelastically** (permanent). Subsidence at the land surface is the compaction summed over all the layers beneath it.

## Set up

Import FloPy to build and run the model, NumPy for array math, and Matplotlib to plot the results. `pathlib.Path` builds the workspace paths.

In [ ]:
import pathlib as pl

import flopy
import matplotlib.pyplot as plt
import numpy as np

## Model parameters

Set the grid, hydraulic properties, and the CSUB/interbed storage properties. The system is four layers: an upper aquifer (layers 1-2), a thin low-permeability **confining unit** (layer 3), and a lower aquifer (layer 4). Cell ids are zero-based, so those are layer indices `0-3`. Lengths are in meters and time in days - MODFLOW has no built-in units, so every input shares those.

The interbed parameters are the heart of this problem. `ssv` is the **inelastic** specific storage (permanent compaction once the effective stress passes the preconsolidation stress) and `sse` is the **elastic** specific storage (recoverable). `kv_delay` is the vertical hydraulic conductivity of the interbed clay - the smaller it is, the longer pore pressure takes to drain and the more a *delay* interbed lags. These values are chosen so the delay and no-delay results differ visibly over the 120-year simulation.

In [ ]:
# Model units
length_units = "meters"
time_units = "days"

# Grid: a 4-layer aquifer system, 20 rows by 15 columns of 2 km cells.
nlay, nrow, ncol = 4, 20, 15
delr = delc = 2000.0
top = 150.0
botm = [50.0, -100.0, -150.0, -350.0]  # layer-bottom elevations (m)
strt = 100.0  # starting and edge (constant) head (m)

# Hydraulic properties by layer. Layer 3 (index 2) is a low-K confining unit
# between the upper aquifer (layers 1-2) and the lower aquifer (layer 4).
icelltype = [1, 0, 0, 0]  # only the top layer is convertible (water table)
k11 = [4.0, 4.0, 0.01, 4.0]  # horizontal hydraulic conductivity (m/day)
sy = [0.3, 0.3, 0.4, 0.3]  # specific yield

# Coarse-grained (aquifer-matrix) CSUB properties, by layer.
gammaw = 9806.65  # unit weight of water (N/m^3)
beta = 4.6612e-10  # compressibility of water (1/Pa)
sgm = [1.77, 1.77, 1.60, 1.77]  # moist specific gravity of sediments
sgs = [2.06, 2.05, 1.94, 2.06]  # saturated specific gravity
cg_theta = [0.32, 0.32, 0.45, 0.32]  # coarse-grained porosity
cg_ske = [1e-6, 1e-6, 1e-6, 1e-6]  # coarse-grained elastic specific storage (1/m)

# Interbeds: compressible fine-grained lenses inside each cell.
ib_thick = [45.0, 70.0, 50.0, 90.0]  # total interbed thickness per cell, by layer (m)
ib_theta = 0.45  # interbed porosity
ssv = 6e-4  # inelastic (virgin) specific storage - permanent compaction (1/m)
sse = 1e-5  # elastic specific storage - recoverable compaction (1/m)
pcs0 = 15.0  # initial preconsolidation-stress offset (m of water)

# Delay-interbed properties (used only when cdelay="delay").
kv_delay = 1e-5  # vertical hydraulic conductivity of the interbed clay (m/day)
ndelaycells = 19  # cells used to resolve flow through each delay interbed

# Stress periods: (0) steady state, (1) 60 years pumping, (2) 60 years recovery.
nper = 3
tdis = [
    (0.0, 1, 1.0),
    (21915.0, 60, 1.0),
    (21915.0, 60, 1.0),
]  # (perlen_days, nstp, tsmult)

# Two pumping wells, active only in stress period 1 (index 1): one in the upper
# aquifer and one in the lower aquifer. cellids are zero-based (layer, row, column).
well_locs = [(1, 8, 9), (3, 11, 6)]
well_q = -72000.0  # withdrawal rate (m^3/day)

## Build the model

Build the whole simulation in one function so the two scenarios differ by a single input. The flow model is standard - discretization (**DIS**), initial conditions (**IC**), node-property flow (**NPF**), storage (**STO**), constant-head edges (**CHD**), and two pumping wells (**WEL**). The subsidence comes from the **CSUB** package: one interbed per active cell, with a `cdelay` column set to either `"nodelay"` or `"delay"`. For delay interbeds we also supply the interbed vertical conductivity (`kv`), an initial interbed head (`h0`), and `ndelaycells`. Each interbed's `packagedata` row is `[number, cellid, cdelay, pcs0, thickness, rnb, ssv, sse, theta, kv, h0, boundname]`.

In [ ]:
def build_csub_simulation(ws, delay):
    """Build the CSUB land-subsidence simulation. Pass delay=True for delay
    interbeds (drainage lags behind the aquifer) or delay=False for no-delay
    interbeds (instantaneous equilibrium); everything else is identical."""
    name = "mf6-csub"
    sim = flopy.mf6.MFSimulation(sim_name=name, sim_ws=str(ws), exe_name="mf6")
    flopy.mf6.ModflowTdis(sim, nper=nper, perioddata=tdis, time_units=time_units)
    flopy.mf6.ModflowIms(
        sim,
        outer_maximum=100,
        inner_maximum=300,
        outer_dvclose=1e-9,
        inner_dvclose=1e-9,
        rcloserecord=1e-6,
        linear_acceleration="bicgstab",
        relaxation_factor=0.97,
    )
    gwf = flopy.mf6.ModflowGwf(
        sim, modelname=name, save_flows=True, newtonoptions="newton"
    )
    flopy.mf6.ModflowGwfdis(
        gwf,
        length_units=length_units,
        nlay=nlay,
        nrow=nrow,
        ncol=ncol,
        delr=delr,
        delc=delc,
        top=top,
        botm=botm,
    )
    flopy.mf6.ModflowGwfic(gwf, strt=strt)
    flopy.mf6.ModflowGwfnpf(
        gwf, icelltype=icelltype, k=k11, save_specific_discharge=True
    )
    flopy.mf6.ModflowGwfsto(
        gwf,
        iconvert=icelltype,
        ss=0.0,
        sy=sy,
        steady_state={0: True},
        transient={1: True},
    )

    # Constant-head cells around the model edge hold the regional water level;
    # pumping in the interior draws it down.
    def edge(i, j):
        return i in (0, nrow - 1) or j in (0, ncol - 1)

    chd = [
        [(k, i, j), strt]
        for k in range(nlay)
        for i in range(nrow)
        for j in range(ncol)
        if edge(i, j)
    ]
    flopy.mf6.ModflowGwfchd(gwf, stress_period_data={0: chd})
    flopy.mf6.ModflowGwfwel(
        gwf,
        stress_period_data={
            1: [[cid, well_q] for cid in well_locs],
            2: [[cid, 0.0] for cid in well_locs],
        },
    )

    # One interbed per active (non-edge) cell. The cdelay column is the only
    # thing that differs between the two scenarios; kv and h0 are ignored for
    # nodelay interbeds (999.0 is a conventional placeholder).
    cdelay = "delay" if delay else "nodelay"
    packagedata = []
    icsub = 0
    for k in range(nlay):
        for i in range(nrow):
            for j in range(ncol):
                if edge(i, j):
                    continue
                kv = kv_delay if delay else 999.0
                h0 = strt if delay else 999.0
                boundname = f"{k + 1:02d}_{i + 1:02d}_{j + 1:02d}"
                packagedata.append(
                    [
                        icsub,
                        (k, i, j),
                        cdelay,
                        pcs0,
                        ib_thick[k],
                        1.0,
                        ssv,
                        sse,
                        ib_theta,
                        kv,
                        h0,
                        boundname,
                    ]
                )
                icsub += 1

    csub_kwargs = dict(
        save_flows=True,
        boundnames=True,
        ninterbeds=len(packagedata),
        cg_ske_cr=cg_ske,
        cg_theta=cg_theta,
        sgm=sgm,
        sgs=sgs,
        beta=beta,
        gammaw=gammaw,
        compaction_filerecord=f"{name}.csub.comp",
        packagedata=packagedata,
    )
    if delay:
        csub_kwargs["ndelaycells"] = ndelaycells
    flopy.mf6.ModflowGwfcsub(gwf, **csub_kwargs)

    flopy.mf6.ModflowGwfoc(
        gwf,
        head_filerecord=f"{name}.hds",
        budget_filerecord=f"{name}.cbc",
        saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
    )
    return sim

## Run both scenarios

Build, write, and run the model twice - once with no-delay interbeds and once with delay interbeds. Each run writes to its own workspace under `models/`. The delay run takes a little longer because CSUB solves a small vertical-flow problem inside every interbed.

In [ ]:
sims = {}
for delay in (False, True):
    label = "delay" if delay else "nodelay"
    ws = pl.Path("./models") / "mf6-csub" / label
    sim = build_csub_simulation(ws, delay)
    sim.write_simulation(silent=True)
    success, buff = sim.run_simulation(silent=True)
    assert success, f"{label} simulation did not converge"
    sims[label] = sim
    print(f"{label} simulation finished")

## How subsidence develops over time

Read the compaction output from each run and sum it over all four layers to get the **subsidence** - the downward movement of the land surface - in every column at every time step. Then compare the two scenarios at the cell that subsides the most.

In [ ]:
def subsidence_over_time(sim):
    """Return (years, subsidence). subsidence has shape (ntimes, nrow, ncol):
    the land-surface lowering in each column, i.e. the compaction summed over
    all layers."""
    gwf = sim.get_model("mf6-csub")
    cobj = gwf.csub.output.compaction()
    comp = cobj.get_alldata()  # (ntimes, nlay, nrow, ncol)
    subsidence = comp.sum(axis=1)  # sum over layers
    years = np.array(cobj.get_times()) / 365.25
    return years, subsidence


years, sub_nodelay = subsidence_over_time(sims["nodelay"])
_, sub_delay = subsidence_over_time(sims["delay"])

# Find the most-affected cell (in the no-delay run) and compare there.
imax = np.unravel_index(np.nanargmax(sub_nodelay[-1]), sub_nodelay[-1].shape)

with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    ax.axvspan(0, 60, color="0.9", label="Pumping period")
    ax.plot(
        years,
        sub_nodelay[:, imax[0], imax[1]],
        color="C0",
        lw=2,
        label="No-delay interbeds",
    )
    ax.plot(
        years, sub_delay[:, imax[0], imax[1]], color="C3", lw=2, label="Delay interbeds"
    )
    ax.set_xlim(0, years[-1])
    ax.set_ylim(bottom=0)
    ax.set_xlabel("Time (years)")
    ax.set_ylabel("Land-surface subsidence (m)")
    ax.set_title(
        f"Subsidence at the most-affected cell (row {imax[0]}, column {imax[1]})"
    )
    ax.legend(loc="lower right")
    plt.show()

**What to look for.** During the 60 years of pumping (shaded), the **no-delay** interbeds compact quickly and reach most of their subsidence, while the **delay** interbeds lag well behind - pore pressure is still draining out of the clay. When pumping stops and heads recover, both curves relax slightly: the *elastic* part of the compaction rebounds, while the *inelastic* part is permanent. The delay interbeds never catch up within the simulation, so they end with noticeably less subsidence.

## Where the land sinks

Map the subsidence at the end of the simulation (120 years) for both scenarios on the same color scale. Both use the same grid, so read it from either model.

In [ ]:
with flopy.plot.styles.USGSMap():
    gwf = sims["nodelay"].get_model("mf6-csub")
    vmax = max(np.nanmax(sub_nodelay[-1]), np.nanmax(sub_delay[-1]))
    fig, axes = plt.subplots(1, 2, figsize=(9, 5), constrained_layout=True)
    for ax, title, sub in [
        (axes[0], "No-delay interbeds", sub_nodelay),
        (axes[1], "Delay interbeds", sub_delay),
    ]:
        pmv = flopy.plot.PlotMapView(model=gwf, ax=ax)
        ca = pmv.plot_array(sub[-1], cmap="viridis_r", vmin=0, vmax=vmax)
        pmv.plot_grid(lw=0.3, color="0.7")
        ax.set_title(title)
        ax.set_xlabel("x (m)")
    axes[0].set_ylabel("y (m)")
    fig.colorbar(ca, ax=axes, shrink=0.7, label="Subsidence at 120 years (m)")
    plt.show()

**What to look for.** Both maps show the same pattern - the land sinks most over the pumped cells and tapers toward the constant-head edges - but the delay case is uniformly shallower, because those interbeds have not finished compacting.

## Maximum subsidence at the end of the simulation

Report the largest subsidence anywhere in the model at the final time step for each scenario, and the difference between them.

In [ ]:
max_nodelay = float(np.nanmax(sub_nodelay[-1]))
max_delay = float(np.nanmax(sub_delay[-1]))
diff = max_nodelay - max_delay

print("Maximum land-surface subsidence at 120 years:")
print(f"  no-delay interbeds : {max_nodelay:6.3f} m")
print(f"  delay interbeds    : {max_delay:6.3f} m")
print(
    f"  difference         : {diff:6.3f} m "
    f"({100 * diff / max_nodelay:.0f}% less with delay interbeds)"
)

**What to look for.** The maximum subsidence with **delay** interbeds is smaller than with **no-delay** interbeds - the delay beds are still draining and have not reached their equilibrium compaction. Given enough time (with heads held down) the two would converge; over a finite pumping history they do not. That is exactly why representing the drainage time of thick, low-permeability interbeds matters when predicting subsidence.

## Exercise

Change one interbed property at a time, re-run both scenarios, and watch the maximum-subsidence assessment:

- Lower `kv_delay` (say to `1e-6`) so the delay interbeds drain even more slowly. Does the gap between delay and no-delay grow or shrink?
- Increase the interbed thickness `ib_thick` or the inelastic storage `ssv`. How does the total subsidence respond?
- Keep pumping in stress period 2 (set the period-2 well rate to `well_q` instead of `0.0`) rather than recovering. Do the delay interbeds eventually catch up to the no-delay result?

Predict the effect first, then check it against the time-series plot and the maximum-subsidence numbers.

## Recap

In this notebook you:

- built a four-layer regional aquifer-system compaction model with the MODFLOW 6 **CSUB** package, adapted from example *ex-gwf-csub-p04*;
- ran it with **no-delay** and **delay** interbeds, changing only the `cdelay` input;
- plotted how land-surface subsidence develops through time and mapped it at the end of the simulation; and
- assessed the **maximum subsidence at the end of the simulation**, finding that delay interbeds - which drain slowly - produce noticeably less subsidence than no-delay interbeds over this pumping history.